In [1]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# Run this cell first
# ============================================================

!pip install transformers sentencepiece tokenizers -q

print("Installation complete!")
print("transformers: provides pre-trained tokenizers")
print("sentencepiece: SentencePiece tokenization library")
print("tokenizers: HuggingFace fast tokenizer library")

Installation complete!
transformers: provides pre-trained tokenizers
sentencepiece: SentencePiece tokenization library
tokenizers: HuggingFace fast tokenizer library


In [2]:
from transformers import GPT2Tokenizer

# ============================================================
# GPT-2 TOKENIZER — uses BPE
# This is the same tokenizer family used by GPT-3
# Vocabulary size: 50,257 tokens
# Trained primarily on English text
# ============================================================

tokenizer_bpe = GPT2Tokenizer.from_pretrained("gpt2")

print(f"GPT-2 BPE Tokenizer")
print(f"Vocabulary size: {tokenizer_bpe.vocab_size:,} tokens")
print()

# ============================================================
# Tokenize various types of text
# Watch how token count varies dramatically
# ============================================================

test_texts = [
    # English — should tokenize very efficiently
    "This is a simple English sentence for testing.",

    # English technical — common technical terms
    "The transformer model uses attention mechanisms for NLP tasks.",

    # Rare English word — should fragment
    "Pneumonoultramicroscopicsilicovolcanoconiosis is a lung disease.",

    # Simple numbers and punctuation
    "The price is $1,234.56 and the date is 2024-01-15.",

    # Repeated simple pattern
    "cat cat cat cat cat cat cat cat cat cat",
]

print("="*65)
print(f"{'Text (truncated)':<45} {'Tokens':<8} {'IDs'}")
print("="*65)

for text in test_texts:
    tokens = tokenizer_bpe.tokenize(text)
    ids = tokenizer_bpe.encode(text)

    # Show the actual token pieces — this reveals the BPE splitting
    print(f"\nText: {text[:60]}")
    print(f"Token pieces: {tokens}")
    print(f"Token count: {len(tokens)}")
    print(f"Token IDs: {ids[:10]}{'...' if len(ids)>10 else ''}")

print()
print(" Notice:")
print("Common English words → single tokens")
print("Rare/long words → split into multiple subword pieces")
print("Numbers and punctuation → often single characters")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GPT-2 BPE Tokenizer
Vocabulary size: 50,257 tokens

Text (truncated)                              Tokens   IDs

Text: This is a simple English sentence for testing.
Token pieces: ['This', 'Ġis', 'Ġa', 'Ġsimple', 'ĠEnglish', 'Ġsentence', 'Ġfor', 'Ġtesting', '.']
Token count: 9
Token IDs: [1212, 318, 257, 2829, 3594, 6827, 329, 4856, 13]

Text: The transformer model uses attention mechanisms for NLP task
Token pieces: ['The', 'Ġtransformer', 'Ġmodel', 'Ġuses', 'Ġattention', 'Ġmechanisms', 'Ġfor', 'ĠN', 'LP', 'Ġtasks', '.']
Token count: 11
Token IDs: [464, 47385, 2746, 3544, 3241, 11701, 329, 399, 19930, 8861]...

Text: Pneumonoultramicroscopicsilicovolcanoconiosis is a lung dise
Token pieces: ['P', 'neum', 'on', 'oult', 'ram', 'icro', 'sc', 'op', 'ics', 'ilic', 'ov', 'ol', 'can', 'ocon', 'iosis', 'Ġis', 'Ġa', 'Ġlung', 'Ġdisease', '.']
Token count: 20
Token IDs: [47, 25668, 261, 25955, 859, 2500, 1416, 404, 873, 41896]...

Text: The price is $1,234.56 and the date is 2024-01-15.
Token pie

In [3]:
from transformers import GPT2Tokenizer, AutoTokenizer
import matplotlib
matplotlib.use('Agg')

# ============================================================
# TOKEN COUNT COMPARISON ACROSS LANGUAGES
# Same meaning, different token counts
# This is the core of the multilingual fragmentation problem
# ============================================================

tokenizer_bpe = GPT2Tokenizer.from_pretrained("gpt2")

# These sentences all mean approximately the same thing:
# "Artificial intelligence is transforming the world of technology"
sentences = {
    "English":    "Artificial intelligence is transforming the world of technology.",
    "Spanish":    "La inteligencia artificial está transformando el mundo de la tecnología.",
    "French":     "L'intelligence artificielle transforme le monde de la technologie.",
    "German":     "Künstliche Intelligenz verändert die Welt der Technologie.",
    "Hindi":      "कृत्रिम बुद्धिमत्ता प्रौद्योगिकी की दुनिया को बदल रही है।",
    "Telugu":     "కృత్రిమ మేధస్సు సాంకేతిక ప్రపంచాన్ని మార్చివేస్తోంది.",
    "Tamil":      "செயற்கை நுண்ணறிவு தொழில்நுட்ப உலகை மாற்றுகிறது.",
    "Arabic":     "يُحوّل الذكاء الاصطناعي عالم التكنولوجيا.",
    "Chinese":    "人工智能正在改变技术世界。",
    "Japanese":   "人工知能は技術の世界を変えています。",
}

print("Token Count Comparison — GPT-2 BPE (English-trained)")
print("Same meaning across different languages")
print("="*70)
print(f"{'Language':<12} {'Token Count':<15} {'Tokens (first 10)'}")
print("-"*70)

english_count = None
for language, sentence in sentences.items():
    tokens = tokenizer_bpe.tokenize(sentence)
    count = len(tokens)

    if language == "English":
        english_count = count

    ratio = count / english_count if english_count else 1.0
    ratio_str = f"({ratio:.1f}x English)" if language != "English" else "(baseline)"

    # Show first few tokens to reveal fragmentation
    first_tokens = str(tokens[:6]).replace("'", "")

    print(f"{language:<12} {count:<6} {ratio_str:<20} {first_tokens}")

print()
print(" Key insight:")
print("English-trained BPE tokenizes English very efficiently")
print("Indian language scripts get fragmented into many small pieces")
print("More tokens = longer sequence = slower inference at every stage")
print()
print("This is WHY Sarvam needs a multilingual-trained tokenizer")
print("Their tokenizer learns Indian language subword patterns")
print("Reducing token counts for Indian languages dramatically")

Token Count Comparison — GPT-2 BPE (English-trained)
Same meaning across different languages
Language     Token Count     Tokens (first 10)
----------------------------------------------------------------------
English      10     (baseline)           [Art, ificial, Ġintelligence, Ġis, Ġtransforming, Ġthe]
Spanish      19     (1.9x English)       [La, Ġintel, igen, cia, Ġartificial, Ġest]
French       17     (1.7x English)       [L, "", intelligence, Ġartific, iel, le]
German       21     (2.1x English)       [K, Ã¼, n, st, lic, he]
Hindi        96     (9.6x English)       [à¤, ķ, à¥, ĥ, à¤, ¤]
Telugu       149    (14.9x English)      [à, °, ķ, à, ±, ĥ]
Tamil        131    (13.1x English)      [à, ®, ļ, à, ¯, Ĩ]
Arabic       42     (4.2x English)       [ÙĬ, Ù, ı, Ø, Ń, ÙĪ]
Chinese      27     (2.7x English)       [äºº, å·, ¥, æ, Ļ, º]
Japanese     29     (2.9x English)       [äºº, å·, ¥, ç, Ł, ¥]

 Key insight:
English-trained BPE tokenizes English very efficiently
Indian language scri

In [4]:
from transformers import AutoTokenizer

# ============================================================
# COMPARE TOKENIZERS: GPT-2 BPE vs LLaMA SentencePiece
# LLaMA's tokenizer has a larger vocabulary (32,000)
# and was trained on more multilingual data
# ============================================================

print("Loading tokenizers...")

# GPT-2 BPE — English-centric, 50k vocab
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")

# LLaMA tokenizer — SentencePiece, 32k vocab, more multilingual
# Using a public model that uses SentencePiece
try:
    tokenizer_llama = AutoTokenizer.from_pretrained("huggyllama/llama-7b")
    llama_available = True
    print(f"GPT-2 vocab size:  {tokenizer_gpt2.vocab_size:,}")
    print(f"LLaMA vocab size:  {tokenizer_llama.vocab_size:,}")
except:
    llama_available = False
    print(f"GPT-2 vocab size: {tokenizer_gpt2.vocab_size:,}")
    print("LLaMA tokenizer not available — showing GPT-2 analysis only")

print()

# ============================================================
# Test sentences in multiple languages
# ============================================================

test_cases = [
    ("English simple",    "The weather is nice today."),
    ("English technical", "The attention mechanism computes query key value products."),
    ("Hindi",             "आज मौसम बहुत अच्छा है।"),
    ("Telugu",            "నేడు వాతావరణం చాలా బాగుంది."),
    ("Tamil",             "இன்று வானிலை மிகவும் நன்றாக உள்ளது."),
    ("Mixed Eng+Hindi",   "The model is processing हिंदी text efficiently."),
]

if llama_available:
    print(f"{'Text':<35} {'GPT2 tokens':<15} {'LLaMA tokens':<15} {'LLaMA/GPT2'}")
    print("-"*75)

    for label, text in test_cases:
        gpt2_tokens = tokenizer_gpt2.tokenize(text)
        llama_tokens = tokenizer_llama.tokenize(text)

        gpt2_count = len(gpt2_tokens)
        llama_count = len(llama_tokens)
        ratio = llama_count / gpt2_count if gpt2_count > 0 else 0

        print(f"{label:<35} {gpt2_count:<15} {llama_count:<15} {ratio:.2f}x")

    print()
    print("What to look for:")
    print("For English: both tokenizers should be similar")
    print("For Indian languages: better multilingual tokenizer = fewer tokens")
    print("Fewer tokens = faster inference = lower latency = more throughput")
else:
    print(f"{'Text':<35} {'GPT2 tokens':<12} {'Actual token pieces'}")
    print("-"*75)

    for label, text in test_cases:
        tokens = tokenizer_gpt2.tokenize(text)
        count = len(tokens)
        # Show the actual fragmentation
        pieces = str(tokens[:8])
        print(f"{label:<35} {count:<12} {pieces}")

    print()
    print(" Notice how Indian language text fragments into many small pieces")
    print("Each fragment = one token = one embedding lookup = more work for transformer")

Loading tokenizers...


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

GPT-2 vocab size:  50,257
LLaMA vocab size:  32,000

Text                                GPT2 tokens     LLaMA tokens    LLaMA/GPT2
---------------------------------------------------------------------------
English simple                      6               6               1.00x
English technical                   10              10              1.00x
Hindi                               35              27              0.77x
Telugu                              73              74              1.01x
Tamil                               95              40              0.42x
Mixed Eng+Hindi                     17              13              0.76x

What to look for:
For English: both tokenizers should be similar
For Indian languages: better multilingual tokenizer = fewer tokens
Fewer tokens = faster inference = lower latency = more throughput


In [5]:
import time
import torch
from transformers import GPT2Tokenizer

# ============================================================
# EXPERIMENT: measure tokenization latency vs inference latency
# This quantifies where time is actually spent
# Shows tokenization is small but token COUNT drives everything else
# ============================================================

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

print("="*65)
print("EXPERIMENT: Tokenization Time vs What Token Count Drives")
print("="*65)

# ============================================================
# Part 1: How long does tokenization itself take?
# ============================================================

test_sentences = [
    "Hello world.",                                          # very short
    "The transformer model processes text using attention.", # medium
    "Artificial intelligence and machine learning are rapidly transforming industries worldwide.",  # longer
]

print("\nPart 1: Raw Tokenization Speed")
print(f"{'Text length (chars)':<22} {'Token count':<14} {'Tokenize time (ms)'}")
print("-"*55)

num_runs = 1000

for sentence in test_sentences:
    # Time the tokenization
    start = time.time()
    for _ in range(num_runs):
        tokens = tokenizer.encode(sentence)
    tokenize_ms = (time.time() - start) / num_runs * 1000

    token_count = len(tokens)
    char_count = len(sentence)

    print(f"{char_count:<22} {token_count:<14} {tokenize_ms:.4f}")

print()
print(" Tokenization itself is very fast (sub-millisecond)")
print("The real impact of token count is on what comes AFTER tokenization")

# ============================================================
# Part 2: How token count affects downstream computation
# We simulate embedding lookup and simple layer computation
# ============================================================

print("\nPart 2: How Token Count Affects GPU Computation Time")
print(f"{'Tokens':<10} {'Embed lookup':<18} {'One attn layer':<20} {'Total sim'}")
print("-"*60)

d_model = 4096       # realistic large model hidden dim
vocab_size = 32000   # realistic vocabulary size
num_heads = 32
head_dim = d_model // num_heads   # 128

# Simulate embedding matrix
embedding_matrix = torch.randn(vocab_size, d_model)

token_counts = [10, 20, 50, 100, 200, 500]
num_runs = 50

for num_tokens in token_counts:
    # Simulate token IDs
    token_ids = torch.randint(0, vocab_size, (num_tokens,))

    # Time embedding lookup (one per token)
    start = time.time()
    for _ in range(num_runs):
        embeddings = embedding_matrix[token_ids]   # shape: [num_tokens x d_model]
    embed_ms = (time.time() - start) / num_runs * 1000

    # Time one attention layer computation (QKᵀV)
    Q = torch.randn(1, num_heads, num_tokens, head_dim)
    K = torch.randn(1, num_heads, num_tokens, head_dim)
    V = torch.randn(1, num_heads, num_tokens, head_dim)

    start = time.time()
    for _ in range(num_runs):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (head_dim ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
    attn_ms = (time.time() - start) / num_runs * 1000

    total_ms = embed_ms + attn_ms
    print(f"{num_tokens:<10} {embed_ms:<18.4f} {attn_ms:<20.4f} {total_ms:.4f}")

print()
print(" Critical observation:")
print("As token count increases, EVERY operation takes longer")
print("Embedding lookup grows linearly with token count")
print("Attention grows quadratically (or linearly with FlashAttention)")
print("This is why token fragmentation directly kills inference throughput")
print()
print("For Sarvam serving Indian languages:")
print("If Telugu uses 3x more tokens than English for same meaning:")
print("→ 3x more embedding lookups")
print("→ 3x more attention computation per layer")
print("→ 3x more MLP computation per layer")
print("→ 3x more KV-cache memory per request")
print("→ 3x more decode steps for same length response")
print("→ Effectively 3x lower throughput and 3x higher latency")
print()
print("A good multilingual tokenizer reduces this fragmentation")
print("Making Indian language inference approach English efficiency")

EXPERIMENT: Tokenization Time vs What Token Count Drives

Part 1: Raw Tokenization Speed
Text length (chars)    Token count    Tokenize time (ms)
-------------------------------------------------------
12                     3              0.0553
53                     8              0.0978
91                     12             0.1208

 Tokenization itself is very fast (sub-millisecond)
The real impact of token count is on what comes AFTER tokenization

Part 2: How Token Count Affects GPU Computation Time
Tokens     Embed lookup       One attn layer       Total sim
------------------------------------------------------------
10         0.4551             2.5262               2.9813
20         0.2244             1.2343               1.4587
50         0.2722             2.5526               2.8248
100        0.7615             7.2676               8.0291
200        1.2045             41.7415              42.9460
500        6.8543             178.6113             185.4656

 Critical obser